### LSTM

In [9]:
import numpy as np
import pandas as pd
from binance.client import Client
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, GRU, Dense, Dropout, BatchNormalization, Flatten
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2
from sklearn.utils.class_weight import compute_class_weight
import warnings

warnings.filterwarnings('ignore')

SYMBOL = "BTCUSDT"
INTERVAL = Client.KLINE_INTERVAL_1HOUR
LIMIT_CANDLES = 15000  
DL_SEQUENCE_LENGTH = 128
# Đã cập nhật Features
DL_FEATURES = ["Log_Return", "High_Low_Spread", "Close_Open_Spread", "Volume_Log", "Volatility_24"]

print(f"🔥 KHỞI ĐỘNG LÒ RÈN ĐỘC TÔN (GRU + DIFF FEATURES) CHO {SYMBOL}")

client = Client() 
print("\n📥 Đang tải dữ liệu lịch sử từ Binance (15,000 nến)...")
klines = client.get_historical_klines(SYMBOL, INTERVAL, "2 years ago UTC")
klines = klines[-LIMIT_CANDLES:]
df = pd.DataFrame(klines, columns=["Open time", "Open", "High", "Low", "Close", "Volume", "Close time", "Quote Asset", "Trades", "Taker Buy Base", "Taker Buy Quote", "Ignore"])
for col in ["Open", "High", "Low", "Close", "Volume", "Taker Buy Base"]:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# ========================================================
# FEATURING ENGINEERING: Chuyển sang Log-Returns và Spreads
# ========================================================
df["Log_Return"] = np.log(df["Close"] / df["Close"].shift(1))
df["High_Low_Spread"] = (df["High"] - df["Low"]) / df["Low"]
df["Close_Open_Spread"] = (df["Close"] - df["Open"]) / df["Open"]
df["Volume_Log"] = np.log1p(df["Volume"]) # Trị giá trị ngoại lai của Volume
df["Volatility_24"] = df["Log_Return"].rolling(24).std()

df = df.dropna().reset_index(drop=True)

# Gắn nhãn
df["Future_Return"] = df["Close"].shift(-1) / df["Close"] - 1.0
df["Target_Long"] = (df["Future_Return"] > 0.001).astype(int)
df["Target_Short"] = (df["Future_Return"] < -0.001).astype(int)
df = df.dropna().reset_index(drop=True)

raw_data = df[DL_FEATURES].values
print("\n⚙️ Đang đóng gói và Scale cục bộ...")
X, y_long, y_short = [], [], []

for i in range(len(raw_data) - DL_SEQUENCE_LENGTH):
    window = raw_data[i : i + DL_SEQUENCE_LENGTH]
    window_mean = np.mean(window, axis=0)
    window_std = np.std(window, axis=0) + 1e-9
    scaled_window = (window - window_mean) / window_std
    X.append(scaled_window)
    y_long.append(df["Target_Long"].iloc[i + DL_SEQUENCE_LENGTH - 1])
    y_short.append(df["Target_Short"].iloc[i + DL_SEQUENCE_LENGTH - 1])

X = np.array(X)
y_long = np.array(y_long)
y_short = np.array(y_short)

split_idx = int(len(X) * 0.8)
X_train, X_val = X[:split_idx], X[split_idx:]
yl_train, yl_val = y_long[:split_idx], y_long[split_idx:]
ys_train, ys_val = y_short[:split_idx], y_short[split_idx:]

# Trọng số
weights_l = compute_class_weight('balanced', classes=np.unique(yl_train), y=yl_train)
class_weight_long = {0: weights_l[0], 1: weights_l[1]}

weights_s = compute_class_weight('balanced', classes=np.unique(ys_train), y=ys_train)
class_weight_short = {0: weights_s[0], 1: weights_s[1]}

# ========================================================
# KIẾN TRÚC CNN-GRU (SỬA ĐỔI GLOBAL POOLING)
# ========================================================
def build_cnn_gru_model(input_shape):
    inputs = Input(shape=input_shape)
    
    x = Conv1D(filters=32, kernel_size=3, padding='causal', activation='relu', kernel_regularizer=l2(0.0005))(inputs)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2)(x)
    x = Dropout(0.3)(x)
    
    # Bỏ return_sequences=True, chỉ lấy hidden state cuối cùng
    x = GRU(32, return_sequences=False, kernel_regularizer=l2(0.0005))(x)
    x = Dropout(0.3)(x)
    
    # Flatten/Dense trực tiếp từ GRU state
    x = Dense(16, activation='relu', kernel_regularizer=l2(0.0005))(x)
    x = Dropout(0.3)(x)
    outputs = Dense(1, activation='sigmoid')(x)
    
    model = Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer=Adam(learning_rate=0.0005), 
        loss='binary_crossentropy', 
        metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
    )
    return model

input_shape = (DL_SEQUENCE_LENGTH, len(DL_FEATURES))

lr_scheduler = ReduceLROnPlateau(monitor='val_auc', mode='max', factor=0.5, patience=4, min_lr=1e-6, verbose=1)
early_stopping = EarlyStopping(monitor='val_auc', mode='max', patience=10, restore_best_weights=True)

print("\n🚀 Bắt đầu rèn Mô hình LONG (CNN-GRU + Diff Features)...")
model_long = build_cnn_gru_model(input_shape)
model_long.fit(
    X_train, yl_train, validation_data=(X_val, yl_val),
    epochs=50, batch_size=128,
    class_weight=class_weight_long,
    callbacks=[early_stopping, lr_scheduler],
    verbose=1
)
model_long.save("lstm_seq128_long.keras")

print("\n🚀 Bắt đầu rèn Mô hình SHORT (CNN-GRU + Diff Features)...")
model_short = build_cnn_gru_model(input_shape)
model_short.fit(
    X_train, ys_train, validation_data=(X_val, ys_val),
    epochs=50, batch_size=128,
    class_weight=class_weight_short,
    callbacks=[early_stopping, lr_scheduler],
    verbose=1
)
model_short.save("lstm_seq128_short.keras")

🔥 KHỞI ĐỘNG LÒ RÈN ĐỘC TÔN (GRU + DIFF FEATURES) CHO BTCUSDT

📥 Đang tải dữ liệu lịch sử từ Binance (15,000 nến)...

⚙️ Đang đóng gói và Scale cục bộ...

🚀 Bắt đầu rèn Mô hình LONG (CNN-GRU + Diff Features)...
Epoch 1/50
93/93 [==============================] - 4s 20ms/step - loss: 0.7463 - accuracy: 0.5043 - auc: 0.5182 - val_loss: 0.7197 - val_accuracy: 0.5892 - val_auc: 0.5557 - lr: 5.0000e-04
Epoch 2/50
93/93 [==============================] - 1s 16ms/step - loss: 0.7339 - accuracy: 0.5197 - auc: 0.5226 - val_loss: 0.7213 - val_accuracy: 0.5418 - val_auc: 0.5514 - lr: 5.0000e-04
Epoch 3/50
93/93 [==============================] - 2s 16ms/step - loss: 0.7299 - accuracy: 0.5028 - auc: 0.5167 - val_loss: 0.7228 - val_accuracy: 0.5088 - val_auc: 0.5538 - lr: 5.0000e-04
Epoch 4/50
93/93 [==============================] - 2s 16ms/step - loss: 0.7233 - accuracy: 0.5106 - auc: 0.5326 - val_loss: 0.7187 - val_accuracy: 0.5061 - val_auc: 0.5621 - lr: 5.0000e-04
Epoch 5/50
93/93 [============